# MatruRaksha — Maternal Health Risk Classifier (Model Building Pipeline)

> **Decision support, not diagnosis.** Trained on the UCI Maternal Health Risk dataset (~1 014 rows).  
> Follows the 14-step model building, training, evaluation, and serialization protocol.

## 1 · Install the extra library
Installs SHAP, the tool that explains each prediction (and gradient boosting libraries).

In [ ]:
# Run once — restart the kernel after the first install if prompted
!pip install shap xgboost lightgbm --quiet

---
## 2 · Imports
Loads all libraries once and sets up workspace directories.

In [ ]:
# ── Standard library ──────────────────────────────────────────────
import warnings, os, sys, json, hashlib, datetime
warnings.filterwarnings("ignore")

# ── Project directory setup ───────────────────────────────────────
CURRENT_DIR = os.path.abspath(os.getcwd())
if os.path.basename(CURRENT_DIR) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR  = os.path.join(PROJECT_ROOT, "data", "raw")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Data & math ───────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# ── scikit-learn ──────────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing  import StandardScaler
from sklearn.pipeline        import Pipeline
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, recall_score, f1_score, accuracy_score
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.base import clone

# ── Gradient boosting ─────────────────────────────────────────────
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier

# ── Explainability ────────────────────────────────────────────────
import shap

# ── Serialisation ─────────────────────────────────────────────────
import joblib

print("✅  All imports successful.")
print(f"📁  Project Root : {PROJECT_ROOT}")
print(f"📁  Data Dir     : {DATA_DIR}")
print(f"📁  Models Dir   : {MODEL_DIR}")

---
## 3 · Upload the dataset
File picker for your CSV (Colab) or auto-detection in `data/raw/` (local).

In [ ]:
CSV_PATH = None  # override with an explicit path if desired

if CSV_PATH is None:
    try:
        from google.colab import files as _colab_files
        print("📂  Pick the CSV file …")
        _uploaded = _colab_files.upload()
        CSV_PATH = list(_uploaded.keys())[0]
    except (ImportError, ModuleNotFoundError):
        candidates = [
            os.path.join(DATA_DIR, "Maternal Health Risk Data Set.csv"),
            os.path.join(PROJECT_ROOT, "Maternal Health Risk Data Set.csv"),
            os.path.join(DATA_DIR, "maternal_health_risk.csv"),
        ]
        if os.path.exists(DATA_DIR):
            for f in os.listdir(DATA_DIR):
                if f.lower().endswith(".csv"):
                    candidates.insert(0, os.path.join(DATA_DIR, f))

        for cand in candidates:
            if os.path.isfile(cand):
                CSV_PATH = cand
                break

        if CSV_PATH is None:
            raise FileNotFoundError(
                f"Dataset not found in {DATA_DIR}.\n"
                "Please place 'Maternal Health Risk Data Set.csv' into data/raw/\n"
                "or set CSV_PATH manually in this cell."
            )

print(f"✅  Using dataset: {CSV_PATH}")

---
## 4 · Settings
Feature order, labels, folds, and the FORCE_MODEL switch.

In [ ]:
# ── Feature order (must match the Flutter form & FastAPI schema) ──
FEATURES = ["Age", "SystolicBP", "DiastolicBP", "BS", "BodyTemp", "HeartRate"]

# ── Label encoding ────────────────────────────────────────────────
LABEL_COL   = "RiskLevel"
LABEL_MAP   = {"low risk": 0, "mid risk": 1, "high risk": 2}
LABEL_NAMES = ["low", "mid", "high"]      # index-aligned with 0, 1, 2
COLORS      = ["#2ecc71", "#f39c12", "#e74c3c"]

# ── Cross-validation ──────────────────────────────────────────────
N_FOLDS     = 5
RANDOM_SEED = 42

# ── Model selection override ──────────────────────────────────────
# Set to "rf", "lr", "xgb", or "lgbm" to force that model.
# Set to None to auto-select by best high-risk recall.
FORCE_MODEL = None

# ── Outlier thresholds ────────────────────────────────────────────
HR_MIN = 40   # drop heart-rate rows below this (physiologically implausible)

print("✅  Settings locked.")
print(f"    Features     : {FEATURES}")
print(f"    Folds        : {N_FOLDS}")
print(f"    Force model  : {FORCE_MODEL or 'auto (best high-risk recall)'}")

---
## 5 · Load and clean the data
Fixes labels, drops impossible heart rates and duplicate rows.

In [ ]:
# ── Load ──────────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)
df_raw.columns = df_raw.columns.str.strip()
print(f"Loaded {len(df_raw)} rows, {df_raw.shape[1]} columns.")
print(f"Columns: {list(df_raw.columns)}\n")

df = df_raw.copy()

# ── 5a. Fix labels ───────────────────────────────────────────────
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
unmapped = set(df[LABEL_COL].unique()) - set(LABEL_MAP.keys())
if unmapped:
    raise ValueError(f"Unexpected labels found: {unmapped}")

df["y"] = df[LABEL_COL].map(LABEL_MAP)
print(f"Label mapping applied: {LABEL_MAP}")

# ── 5b. Drop impossible heart rates ──────────────────────────────
bad_hr = df["HeartRate"] < HR_MIN
n_bad_hr = bad_hr.sum()
if n_bad_hr:
    print(f"\n⚠️  Dropping {n_bad_hr} rows with HeartRate < {HR_MIN} bpm:")
    print(df.loc[bad_hr, ["HeartRate", LABEL_COL]].value_counts().to_string())
    df = df[~bad_hr].copy()
else:
    print(f"\n✓  No heart-rate outliers below {HR_MIN} bpm.")

# ── 5c. Drop exact duplicates ────────────────────────────────────
n_before = len(df)
df = df.drop_duplicates(subset=FEATURES + [LABEL_COL]).reset_index(drop=True)
n_dupes = n_before - len(df)
print(f"\n🗑  Dropped {n_dupes} exact duplicate rows.")

# ── 5d. Null check ───────────────────────────────────────────────
null_counts = df[FEATURES].isnull().sum()
if null_counts.any():
    print(f"\n⚠️  Nulls found:\n{null_counts[null_counts > 0]}")
    df = df.dropna(subset=FEATURES + ["y"]).reset_index(drop=True)
else:
    print("✓  No nulls in feature columns.")

print(f"\n✅  Clean dataset: {len(df)} rows.")

---
## 6 · Look at the data, build X and y
Class counts and summary statistics.

In [ ]:
# ── Class counts ─────────────────────────────────────────────────
class_counts = df["y"].value_counts().reindex([0, 1, 2], fill_value=0)
class_counts.index = LABEL_NAMES
print("Class distribution:")
print(class_counts.to_string())
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(LABEL_NAMES, class_counts.values, color=COLORS, edgecolor="black")
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")
axes[0].set_ylim(0, max(class_counts.values) * 1.15)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 3, str(v), ha="center", fontweight="bold")

axes[1].pie(class_counts.values, labels=LABEL_NAMES, colors=COLORS,
            autopct="%1.1f%%", startangle=90, wedgeprops={"edgecolor": "black"})
axes[1].set_title("Class Proportions")
plt.tight_layout()
plt.show()

# ── Summary statistics ───────────────────────────────────────────
print("\nSummary statistics (cleaned data):")
display_df = df[FEATURES].describe().round(2)
print(display_df.to_string())

# ── Build X and y ────────────────────────────────────────────────
X = df[FEATURES].values.astype(np.float64)
y = df["y"].values.astype(np.int32)

print(f"\n✅  X shape: {X.shape}, y shape: {y.shape}")
print(f"    y distribution: { {LABEL_NAMES[i]: int(c) for i, c in enumerate(np.bincount(y, minlength=3))} }")

---
## 7 · Define the four models
Logistic Regression, Random Forest, XGBoost, LightGBM.

In [ ]:
# ── Logistic Regression (with StandardScaler Pipeline) ────────────
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(
                   C=1.0,
                   class_weight="balanced",
                   solver="lbfgs",
                   max_iter=2000,
                   random_state=RANDOM_SEED
               ))
])

# ── Random Forest ────────────────────────────────────────────────
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

# ── XGBoost ──────────────────────────────────────────────────────
xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="mlogloss",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

# ── LightGBM ─────────────────────────────────────────────────────
lgbm_clf = LGBMClassifier(
    n_estimators=300,
    max_depth=-1,
    learning_rate=0.1,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    verbose=-1,
    n_jobs=-1
)

# ── Collect into a dict for easy iteration ───────────────────────
MODELS = {
    "lr":   ("Logistic Regression", lr_pipe),
    "rf":   ("Random Forest",       rf_clf),
    "xgb":  ("XGBoost",             xgb_clf),
    "lgbm": ("LightGBM",            lgbm_clf),
}

print("✅  Four models defined:\n")
for key, (name, _) in MODELS.items():
    print(f"    {key:5s} → {name}")

---
## 8 · Helper functions
Training, stratified cross-validation, and scoring (recall per class).

In [ ]:
def evaluate_model_cv(model_key, model_name, estimator, X, y, n_folds=N_FOLDS, random_seed=RANDOM_SEED):
    """
    Performs stratified K-fold cross validation and computes recall per class,
    accuracy, and macro/weighted F1 score across each fold.
    """
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_seed)
    fold_metrics = []
    all_y_true, all_y_pred, all_y_proba = [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        clf = clone(estimator)

        # Handle XGBoost balanced class weights
        if model_key == 'xgb':
            sw = compute_sample_weight('balanced', y_tr)
            clf.fit(X_tr, y_tr, sample_weight=sw)
        else:
            clf.fit(X_tr, y_tr)

        y_pred  = clf.predict(X_val)
        y_proba = clf.predict_proba(X_val)
        rec_per_class = recall_score(y_val, y_pred, average=None, labels=[0, 1, 2])

        fold_metrics.append({
            'fold':        fold_idx,
            'accuracy':    accuracy_score(y_val, y_pred),
            'macro_f1':    f1_score(y_val, y_pred, average='macro'),
            'weighted_f1': f1_score(y_val, y_pred, average='weighted'),
            'recall_low':  rec_per_class[0],
            'recall_mid':  rec_per_class[1],
            'recall_high': rec_per_class[2],
        })

        all_y_true.extend(y_val)
        all_y_pred.extend(y_pred)
        all_y_proba.append(y_proba)

    return fold_metrics, (np.array(all_y_true), np.array(all_y_pred), np.vstack(all_y_proba))

print("✅  CV helper functions defined.")

---
## 9 · Run cross-validation and compare
The results table, sorted by high-risk recall.

In [ ]:
cv_results = {}
cv_preds   = {}

for key, (name, estimator) in MODELS.items():
    print(f"Evaluating {name:22s} ...", end=" ")
    metrics, preds = evaluate_model_cv(key, name, estimator, X, y)
    cv_results[key] = metrics
    cv_preds[key]   = preds
    df_f = pd.DataFrame(metrics)
    print(f"High-Risk Recall: {df_f['recall_high'].mean():.3f} ± {df_f['recall_high'].std():.3f}")

# ── Aggregate results and SORT by high-risk recall descending ──────
summary_rows = []
for key, folds in cv_results.items():
    df_folds = pd.DataFrame(folds)
    means = df_folds.drop(columns='fold').mean()
    stds  = df_folds.drop(columns='fold').std()
    row = {'model': MODELS[key][0], 'key': key}
    for col in means.index:
        row[col]          = round(means[col], 3)
        row[col + '_std'] = round(stds[col], 3)
    summary_rows.append(row)

df_compare = pd.DataFrame(summary_rows).sort_values(by='recall_high', ascending=False).set_index('key')

print("\n📊  Results Table (Sorted by High-Risk Recall):\n")
display_cols = ['model', 'recall_high', 'recall_high_std', 'macro_f1', 'accuracy', 'recall_mid', 'recall_low']
print(df_compare[display_cols].to_string())

# Bar chart visualization
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(df_compare['model'], df_compare['recall_high'], yerr=df_compare['recall_high_std'],
              capsize=5, color=['#e74c3c', '#3498db', '#2ecc71', '#9b59b6'], edgecolor='black', linewidth=0.7)
ax.set_ylabel('Recall (High-Risk Class)')
ax.set_title('Model Comparison — High-Risk Recall (Sorted)', fontweight='bold')
ax.set_ylim(0, 1.15)
for bar, val in zip(bars, df_compare['recall_high']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{val:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10 · Choose the best model
Picks the winner, or forces Random Forest if you ask.

In [ ]:
if FORCE_MODEL:
    best_key = FORCE_MODEL.lower().strip()
    print(f"⚙️  FORCE_MODEL is set → selected '{best_key}'.")
else:
    best_key = df_compare.index[0]  # top row of sorted comparison table
    print(f"🏆  Auto-selected by highest high-risk recall → '{best_key}'.")

best_name, best_base = MODELS[best_key]
print(f"    Model Name       : {best_name}")
print(f"    High-Risk Recall : {df_compare.loc[best_key, 'recall_high']:.3f} ± {df_compare.loc[best_key, 'recall_high_std']:.3f}")
print(f"    Macro F1         : {df_compare.loc[best_key, 'macro_f1']:.3f}")

---
## 11 · Detailed report
Precision and recall per class, plus the confusion matrix.

In [ ]:
y_true, y_pred, y_proba = cv_preds[best_key]

print(f"{'═'*60}")
print(f"  Detailed Evaluation Report: {best_name} (Pooled Out-of-Fold)")
print(f"{'═'*60}\n")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=3))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title(f"Confusion Matrix — {best_name}", fontweight="bold")
plt.grid(False)
plt.tight_layout()
plt.show()

---
## 12 · Train the final model and save it
Writes `model.joblib` and `meta.json`.

In [ ]:
# ── Retrain on the complete dataset ──────────────────────────────
final_model = clone(best_base)
if best_key == 'xgb':
    sw_full = compute_sample_weight('balanced', y)
    final_model.fit(X, y, sample_weight=sw_full)
else:
    final_model.fit(X, y)

# ── Write model.joblib ───────────────────────────────────────────
model_path = os.path.join(MODEL_DIR, "model.joblib")
joblib.dump(final_model, model_path)
print(f"💾  Saved final model → {model_path}")

# Also save with specific version name for provenance
versioned_path = os.path.join(MODEL_DIR, f"{best_key}_maternal_risk_v1.joblib")
joblib.dump(final_model, versioned_path)

# ── Write meta.json ──────────────────────────────────────────────
data_hash = hashlib.sha256(df[FEATURES + [LABEL_COL]].to_csv(index=False).encode()).hexdigest()[:16]

meta = {
    'model_key':       best_key,
    'model_name':      best_name,
    'model_file':      "model.joblib",
    'features':        FEATURES,
    'label_map':       LABEL_MAP,
    'label_names':     LABEL_NAMES,
    'n_folds':         N_FOLDS,
    'n_samples':       int(len(X)),
    'dataset_hash':    data_hash,
    'cv_recall_high':  float(df_compare.loc[best_key, 'recall_high']),
    'cv_macro_f1':     float(df_compare.loc[best_key, 'macro_f1']),
    'cv_accuracy':     float(df_compare.loc[best_key, 'accuracy']),
    'trained_at':      datetime.datetime.now().isoformat(),
    'random_seed':     RANDOM_SEED,
}

meta_path = os.path.join(MODEL_DIR, "meta.json")
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f"💾  Saved metadata    → {meta_path}")

---
## 13 · Which vitals matter overall
Feature importance chart, good for a slide.

In [ ]:
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
elif hasattr(final_model, 'named_steps') and hasattr(final_model.named_steps.get('clf'), 'coef_'):
    importances = np.abs(final_model.named_steps['clf'].coef_).mean(axis=0)
else:
    importances = np.ones(len(FEATURES))

sort_idx = np.argsort(importances)[::-1]
sorted_features = [FEATURES[i] for i in sort_idx]
sorted_importances = importances[sort_idx]

plt.figure(figsize=(9, 4.5))
plt.barh(sorted_features[::-1], sorted_importances[::-1],
         color='#3498db', edgecolor='black', linewidth=0.6)
plt.xlabel('Relative Feature Importance', fontsize=11)
plt.title(f'Which Vitals Matter Overall — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'feature_importance_slide.png'), dpi=200)
plt.show()

print("\nRanking of Vitals (most to least impactful):")
for rank, (feat, imp) in enumerate(zip(sorted_features, sorted_importances), 1):
    print(f"  {rank}. {feat:15s} : {imp:.4f}")

---
## 14 · Predict and name the top factors
The function your API will use.

In [ ]:
def predict_vitals(age, systolic, diastolic, blood_sugar, body_temp, heart_rate,
                   model=final_model, feature_names=FEATURES, label_names=LABEL_NAMES):
    """
    API Prediction Function:
    Takes 6 visit vitals, returns predicted risk class, probabilities, and top driving factors.
    """
    vitals_array = np.array([[age, systolic, diastolic, blood_sugar, body_temp, heart_rate]], dtype=np.float64)

    # Inference
    pred_idx = int(model.predict(vitals_array)[0])
    probs = model.predict_proba(vitals_array)[0]

    # Driving factors
    if hasattr(model, 'feature_importances_'):
        feat_weights = model.feature_importances_
    elif hasattr(model, 'named_steps') and hasattr(model.named_steps.get('clf'), 'coef_'):
        feat_weights = np.abs(model.named_steps['clf'].coef_[pred_idx])
    else:
        feat_weights = np.ones(len(feature_names))

    top_factors = [feature_names[i] for i in np.argsort(feat_weights)[::-1][:2]]

    return {
        "predicted_class": label_names[pred_idx],
        "risk_rank":       pred_idx + 1,  # 1=low, 2=mid, 3=high
        "probabilities":   {label_names[i]: round(float(p), 4) for i, p in enumerate(probs)},
        "top_factors":     top_factors,
        "model_version":   "rf-v1" if "Random" in str(type(model)) else "v1",
    }


# ── Test inference with a sample case ─────────────────────────────
sample_case = {
    "age": 24,
    "systolic": 142,
    "diastolic": 92,
    "blood_sugar": 7.8,
    "body_temp": 99.1,
    "heart_rate": 88
}

api_result = predict_vitals(**sample_case)

print("\n🩺  Sample API Inference Result:")
print(json.dumps(api_result, indent=2))

print("\n✅  Step 14 complete: Prediction function is ready for FastAPI.")